# 2.2 — Unconstrained Optimality Conditions

Unconstrained optimality conditions turn local calculus into an optimizer's checklist: if the gradient is nonzero, a downhill direction exists; if the gradient is zero, curvature decides whether the point is a minimum, maximum, or saddle. In this lesson, you will build that checklist from first principles with NumPy, inspect the numbers that justify each decision, and see why convexity upgrades local evidence into a global guarantee.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build unconstrained optimality one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the gradient, Hessian, and eigenvalue test, is derived and shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, grids, finite differences, and linear algebra.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any sampled starting points.

### 1. Nonzero gradient means a local descent direction exists

A differentiable objective can be inspected locally by its first-order approximation. Near a point $x$, a tiny move $s$ changes the function by roughly $f(x+s)-f(x)\approx \nabla f(x)^\top s$. If the gradient is nonzero, choosing $s=-\eta\nabla f(x)$ makes this approximation negative: the dot product becomes $-\eta\lVert\nabla f(x)\rVert^2$. That is why the negative gradient is the first descent direction used by gradient descent.

In [ ]:
x_w = np.array([2.0, -1.0])  # current point for f(x,y) = (x-1)^2 + 2(y+2)^2.
grad_w = np.array([2 * (x_w[0] - 1), 4 * (x_w[1] + 2)])  # analytic gradient at x_w.
eta_w = 0.1  # a small local step size.
step_w = -eta_w * grad_w  # first-order downhill move.
print("point:", x_w)
print("gradient:", grad_w)
print("descent step:", step_w)
assert np.allclose(grad_w, [2.0, 4.0])

▶ What you'll see: the gradient is nonzero, so the proposed move points opposite to it.

In [ ]:
def f1_w(z):
    return (z[0] - 1) ** 2 + 2 * (z[1] + 2) ** 2

old_val_w = f1_w(x_w)
new_val_w = f1_w(x_w + step_w)
linear_change_w = float(grad_w @ step_w)
print("f before:", round(old_val_w, 3), "f after:", round(new_val_w, 3))
print("linearized change grad·step:", round(linear_change_w, 3))
assert new_val_w < old_val_w and round(linear_change_w, 3) == -2.0

▶ What you'll see: the actual function value decreases, and the local linear prediction is negative.

In [ ]:
plt.figure(figsize=(4.5, 3.4))
plt.quiver([0], [0], [grad_w[0]], [grad_w[1]], angles="xy", scale_units="xy", scale=1, color="crimson", label="gradient")
plt.quiver([0], [0], [step_w[0]], [step_w[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="-η gradient")
plt.xlim(-1, 3); plt.ylim(-1, 5); plt.axhline(0, color="gray", linewidth=0.6); plt.axvline(0, color="gray", linewidth=0.6)
plt.title("1: gradient vs descent step"); plt.legend(); plt.show()

▶ What you'll see: the descent step points exactly opposite the gradient, the direction of steepest local increase.

*Why it's done this way: the first-order Taylor model is the cheapest trustworthy local model; choosing the negative gradient guarantees the model decreases because $\nabla f^\top(-\eta\nabla f)$ is strictly negative whenever the gradient is nonzero.*

### 2. Zero gradient is necessary, not sufficient

At an interior smooth minimum, every tiny direction must fail to reduce the objective. That forces the first-order term $\nabla f(x^\star)^\top s$ to be zero for every direction $s$, so $\nabla f(x^\star)=0$. But this only creates a **candidate**. A hilltop and a saddle can also have zero first-order signal, so stationarity must be followed by a curvature check.

In [ ]:
xs_w = np.linspace(-2.0, 2.0, 201)  # inspect three functions on the same 1-D grid.
f_min_w = (xs_w - 1) ** 2 + 2  # true bowl-shaped minimum at x=1.
f_max_w = -xs_w ** 2  # hilltop maximum at x=0.
f_flat_w = xs_w ** 3  # stationary inflection at x=0.
print("candidate locations:", {"bowl": 1.0, "hill": 0.0, "cubic": 0.0})

▶ What you'll see: all three examples have a point where the slope formula will vanish.

In [ ]:
grad_min_at_1_w = 2 * (1.0 - 1.0)
grad_max_at_0_w = -2 * 0.0
grad_cubic_at_0_w = 3 * 0.0 ** 2
print("gradients at candidates:", grad_min_at_1_w, grad_max_at_0_w, grad_cubic_at_0_w)
assert grad_min_at_1_w == grad_max_at_0_w == grad_cubic_at_0_w == 0.0

▶ What you'll see: the gradient test alone accepts a minimum, a maximum, and an inflection point.

In [ ]:
plt.figure(figsize=(5, 3.3))
plt.plot(xs_w, f_min_w, label="(x-1)^2+2")
plt.plot(xs_w, f_max_w, label="-x^2")
plt.plot(xs_w, f_flat_w, label="x^3")
plt.scatter([1, 0, 0], [2, 0, 0], color="black", zorder=3)
plt.axhline(0, color="gray", linewidth=0.6); plt.legend(); plt.title("2: three stationary candidates"); plt.show()

▶ What you'll see: the same zero-slope signal corresponds to three different local shapes.

*Why it's done this way: stationarity is a necessary filter, not a verdict; it tells us the first-order descent certificate is gone, so the next useful local information must come from second-order curvature.*

### 3. In one dimension, the second derivative classifies local shape

In 1-D, curvature is the second derivative. Near a stationary point $x^\star$, Taylor's formula says $f(x^\star+h)\approx f(x^\star)+\frac12 f''(x^\star)h^2$. Since $h^2\ge 0$, positive curvature lifts nearby points above the candidate, negative curvature pushes nearby points below it, and zero curvature leaves the second-order test inconclusive.

In [ ]:
candidates_w = np.array([1.0, 0.0, 0.0])
second_derivs_w = np.array([2.0, -2.0, 0.0])  # for bowl, hill, cubic at their stationary points.
labels_w = np.array(["strict local min", "strict local max", "inconclusive"])
for name_w, curv_w, label_w in zip(["bowl", "hill", "cubic"], second_derivs_w, labels_w):
    print(name_w, "f'' =", curv_w, "->", label_w)
assert second_derivs_w.tolist() == [2.0, -2.0, 0.0]

▶ What you'll see: positive, negative, and zero curvature produce three different classifications.

In [ ]:
h_w = np.array([-0.2, -0.1, 0.1, 0.2])
quad_changes_w = 0.5 * second_derivs_w[:, None] * h_w[None, :] ** 2
print("Taylor second-order changes by row:\n", np.round(quad_changes_w, 4))

▶ What you'll see: the bowl row is positive for every nonzero $h$, the hill row is negative, and the cubic row is all zero.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["bowl", "hill", "cubic"], second_derivs_w, color=["seagreen", "crimson", "gray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("second derivative at candidate"); plt.title("3: curvature sign test"); plt.show()

▶ What you'll see: the sign of $f''$ is the 1-D curvature verdict when the gradient is zero.

*Why it's done this way: after stationarity kills the linear term, the quadratic term is the first nonzero local model; its sign tells whether all sufficiently small moves raise or lower the objective.*

### 4. In many dimensions, Hessian eigenvalues are curvature by direction

For $f:\mathbb{R}^d\to\mathbb{R}$, the Hessian matrix collects all second derivatives. At a stationary point, the local quadratic model is $f(x^\star+s)\approx f(x^\star)+\frac12s^\top Hs$. Eigenvectors reveal special directions where this quadratic form becomes $\lambda\lVert s\rVert^2$, so eigenvalue signs classify the point: all positive means bowl, all negative means peak, mixed signs mean saddle.

In [ ]:
H_saddle_w = np.array([[2.0, 0.0], [0.0, -2.0]])  # Hessian of x^2 - y^2.
evals_w, evecs_w = np.linalg.eigh(H_saddle_w)  # symmetric eigen-decomposition.
print("Hessian:\n", H_saddle_w)
print("eigenvalues:", evals_w)
assert np.allclose(evals_w, [-2.0, 2.0])

▶ What you'll see: one positive eigenvalue and one negative eigenvalue.

In [ ]:
dirs_w = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0] / np.sqrt(2)])
curvatures_w = np.array([float(v_w @ H_saddle_w @ v_w) for v_w in dirs_w])
print("directional curvatures:", np.round(curvatures_w, 3))
assert np.allclose(np.round(curvatures_w, 3), [2.0, -2.0, 0.0])

▶ What you'll see: the x-direction curves up, the y-direction curves down, and a diagonal balances out.

In [ ]:
grid_w = np.linspace(-1.5, 1.5, 101)
X_w, Y_w = np.meshgrid(grid_w, grid_w)
Z_w = X_w ** 2 - Y_w ** 2
plt.figure(figsize=(4.5, 3.6))
plt.contour(X_w, Y_w, Z_w, levels=15, cmap="coolwarm")
plt.scatter([0], [0], color="black")
plt.title("4: saddle contours for x² - y²"); plt.xlabel("x"); plt.ylabel("y"); plt.axis("equal"); plt.show()

▶ What you'll see: contours rise in one direction and fall in the perpendicular direction, the geometry of a saddle.

*Why it's done this way: eigenvalues reduce every possible direction to a finite curvature summary; positive definite Hessian means $s^\top Hs>0$ for every nonzero move, while an indefinite Hessian exposes a downhill direction even at zero gradient.*

### 5. Convexity turns local conditions into global guarantees

For a convex differentiable function, the tangent plane is a global lower bound: $f(y)\ge f(x)+\nabla f(x)^\top(y-x)$. If $\nabla f(x^\star)=0$, the bound becomes $f(y)\ge f(x^\star)$ for every $y$, so the stationary point is globally optimal. Without convexity, a positive-definite Hessian at one stationary point only proves a local minimum.

In [ ]:
xs_conv_w = np.linspace(-1.0, 3.0, 201)
f_conv_w = (xs_conv_w - 1.0) ** 2 + 2.0
x_star_w = 1.0
f_star_w = (x_star_w - 1.0) ** 2 + 2.0
grad_star_w = 2 * (x_star_w - 1.0)
print("x*:", x_star_w, "f(x*):", f_star_w, "gradient:", grad_star_w)
assert grad_star_w == 0.0 and f_star_w == 2.0

▶ What you'll see: the convex bowl has a stationary point at $x=1$.

In [ ]:
global_gap_w = f_conv_w - f_star_w
print("minimum sampled gap:", round(float(np.min(global_gap_w)), 6))
assert np.min(global_gap_w) >= -1e-12

▶ What you'll see: every sampled point has objective value at least as large as the stationary point.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(xs_conv_w, f_conv_w, color="seagreen")
plt.axhline(f_star_w, color="black", linestyle="--", label="global lower value")
plt.scatter([x_star_w], [f_star_w], color="black")
plt.title("5: convex stationary point is global"); plt.legend(); plt.show()

▶ What you'll see: the entire convex curve sits above the stationary value.

*Why it's done this way: convexity supplies a global inequality, not just a local Taylor model; when the tangent slope is zero, that inequality certifies no point anywhere can be lower.*

### 6. Tiny gradients can be misleading without scale and curvature

A small gradient is not the same as optimality. In flat regions, the slope can be tiny even far from the minimizer, while curvature and objective decrease still matter. Practical optimizers therefore check more than $\lVert\nabla f\rVert$: they monitor objective values, step sizes, curvature, and whether stationarity is paired with the right Hessian sign.

In [ ]:
x_flat_w = 10.0
grad_flat_w = 2e-4 * x_flat_w  # gradient of f(x)=1e-4 x^2 at x=10.
f_flat_far_w = 1e-4 * x_flat_w ** 2
f_flat_opt_w = 0.0
print("gradient norm:", abs(grad_flat_w))
print("objective gap:", f_flat_far_w - f_flat_opt_w)
assert abs(grad_flat_w) == 0.002 and round(f_flat_far_w, 3) == 0.01

▶ What you'll see: the gradient is tiny, but the point is still not the minimizer.

In [ ]:
xs_flat_w = np.linspace(-12, 12, 200)
vals_flat_w = 1e-4 * xs_flat_w ** 2
plt.figure(figsize=(4.5, 3))
plt.plot(xs_flat_w, vals_flat_w, color="purple")
plt.scatter([x_flat_w, 0], [f_flat_far_w, 0], color=["red", "black"])
plt.title("6: flat bowl makes gradients small far away"); plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the curve is extremely shallow, so slope alone understates distance from the optimum.

*Why it's done this way: numerical stopping rules need scale awareness because first-order information can be weak in flat regions; optimality conditions are mathematical certificates, while approximate optimization needs several diagnostics.*

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · A nonzero gradient gives a descent step

When the gradient is nonzero, a small step in the negative-gradient direction has negative first-order
change and should lower a smooth objective.

In [ ]:
import numpy as np                              # arrays, gradients, and checks.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t1_x = np.array([3.0, -1.0])                    # -> [3.0, -1.0]
print("point:", t1_x.tolist())                  # -> [3.0, -1.0]
t1_grad = np.array([2.0 * (t1_x[0] - 1.0), 4.0 * (t1_x[1] + 2.0)])  # -> [4.0, 4.0]
print("gradient:", t1_grad.tolist())            # -> [4.0, 4.0]
t1_eta = 0.1                                    # -> 0.1
print("eta:", t1_eta)                           # -> 0.1
t1_step = -t1_eta * t1_grad                     # -> [-0.4, -0.4]
print("descent step:", t1_step.tolist())        # -> [-0.4, -0.4]
t1_next = t1_x + t1_step                        # -> [2.6, -1.4]
print("next point:", t1_next.tolist())          # -> [2.6, -1.4]
t1_f0 = (t1_x[0] - 1.0) ** 2 + 2.0 * (t1_x[1] + 2.0) ** 2          # -> 6.0
print("f before:", t1_f0)                       # -> 6.0
t1_f1 = (t1_next[0] - 1.0) ** 2 + 2.0 * (t1_next[1] + 2.0) ** 2    # -> 3.28
print("f after:", round(float(t1_f1), 3))        # -> 3.28
t1_linear = float(t1_grad @ t1_step)            # -> -3.2
print("linearized change:", round(t1_linear, 3)) # -> -3.2
assert t1_f1 < t1_f0 and round(t1_linear, 3) == -3.2

plt.figure(figsize=(4.2, 3.2))
plt.quiver([t1_x[0]], [t1_x[1]], [t1_grad[0]], [t1_grad[1]], angles="xy", scale_units="xy", scale=1, color="crimson", label="gradient")
plt.quiver([t1_x[0]], [t1_x[1]], [t1_step[0]], [t1_step[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="descent step")
plt.scatter([t1_x[0], t1_next[0]], [t1_x[1], t1_next[1]], color=["black", "seagreen"], zorder=3)
plt.xlim(2.0, 7.5)
plt.ylim(-1.8, 3.6)
plt.title("Toy 1 · step opposes the gradient")
plt.xlabel("x0")
plt.ylabel("x1")
plt.legend()
plt.show()

▶ What you'll see: the green step points opposite the red gradient and lowers the objective.

### ✍️ Toy 2 · Stationarity is only a candidate test

A zero derivative can occur at a minimum, a maximum, or a saddle-like inflection. The first-order test
finds candidates but does not classify them.

In [ ]:
import numpy as np                              # arrays and one-dimensional samples.

t2_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t2_h = 0.5                                      # -> 0.5
print("neighbor spacing:", t2_h)                # -> 0.5
t2_neighbors = np.array([-t2_h, 0.0, t2_h])     # -> [-0.5, 0.0, 0.5]
print("neighbor points:", t2_neighbors.tolist()) # -> [-0.5, 0.0, 0.5]
t2_bowl = t2_neighbors ** 2                     # -> [0.25, 0.0, 0.25]
print("bowl values:", t2_bowl.tolist())         # -> [0.25, 0.0, 0.25]
t2_hill = -(t2_neighbors ** 2)                  # -> [-0.25, -0.0, -0.25]
print("hill values:", t2_hill.tolist())         # -> [-0.25, -0.0, -0.25]
t2_cubic = t2_neighbors ** 3                    # -> [-0.125, 0.0, 0.125]
print("cubic values:", t2_cubic.tolist())       # -> [-0.125, 0.0, 0.125]
t2_gradients = np.array([0.0, 0.0, 0.0])        # -> [0.0, 0.0, 0.0]
print("gradients at zero:", t2_gradients.tolist()) # -> [0.0, 0.0, 0.0]
t2_grid = np.linspace(-1.5, 1.5, 7)             # -> [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
print("plot grid:", t2_grid.tolist())           # -> [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
t2_grid_bowl = t2_grid ** 2                     # -> [2.25, 1.0, 0.25, 0.0, 0.25, 1.0, 2.25]
print("grid bowl:", t2_grid_bowl.tolist())      # -> [2.25, 1.0, 0.25, 0.0, 0.25, 1.0, 2.25]
t2_grid_hill = -(t2_grid ** 2)                  # -> [-2.25, -1.0, -0.25, -0.0, -0.25, -1.0, -2.25]
print("grid hill:", t2_grid_hill.tolist())      # -> [-2.25, -1.0, -0.25, -0.0, -0.25, -1.0, -2.25]
t2_grid_cubic = t2_grid ** 3                    # -> [-3.375, -1.0, -0.125, 0.0, 0.125, 1.0, 3.375]
print("grid cubic:", t2_grid_cubic.tolist())    # -> [-3.375, -1.0, -0.125, 0.0, 0.125, 1.0, 3.375]
assert np.all(t2_gradients == 0.0) and t2_bowl[1] == t2_hill[1] == t2_cubic[1] == 0.0

plt.figure(figsize=(4.8, 3.2))
plt.plot(t2_grid, t2_grid_bowl, marker="o", label="x²")
plt.plot(t2_grid, t2_grid_hill, marker="s", label="-x²")
plt.plot(t2_grid, t2_grid_cubic, marker="^", label="x³")
plt.scatter([0.0, 0.0, 0.0], [0.0, 0.0, 0.0], color="black", zorder=3)
plt.title("Toy 2 · zero slope, different shapes")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

▶ What you'll see: all three curves are stationary at zero, but their nearby shapes disagree.

### ✍️ Toy 3 · Second derivatives classify 1-D shape

After stationarity removes the linear term, the sign of the second derivative decides whether small
moves raise, lower, or fail to decide the local model.

In [ ]:
import numpy as np                              # arrays and Taylor changes.

t3_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t3_curvatures = np.array([2.0, -2.0, 0.0])      # -> [2.0, -2.0, 0.0]
print("second derivatives:", t3_curvatures.tolist()) # -> [2.0, -2.0, 0.0]
t3_h = np.array([-0.5, 0.5])                    # -> [-0.5, 0.5]
print("small moves:", t3_h.tolist())            # -> [-0.5, 0.5]
t3_h_squared = t3_h ** 2                        # -> [0.25, 0.25]
print("h squared:", t3_h_squared.tolist())      # -> [0.25, 0.25]
t3_changes = 0.5 * t3_curvatures[:, None] * t3_h_squared[None, :]  # -> [[0.25, 0.25], [-0.25, -0.25], [0.0, 0.0]]
print("quadratic changes:", t3_changes.tolist()) # -> [[0.25, 0.25], [-0.25, -0.25], [0.0, 0.0]]
t3_labels = np.array(["min", "max", "inconclusive"])  # -> ['min', 'max', 'inconclusive']
print("classifications:", t3_labels.tolist())   # -> ['min', 'max', 'inconclusive']
assert t3_changes[0, 0] > 0.0 and t3_changes[1, 0] < 0.0 and t3_changes[2, 0] == 0.0

plt.figure(figsize=(4.4, 2.8))
plt.bar(t3_labels, t3_curvatures, color=["seagreen", "crimson", "gray"])
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 3 · curvature sign test")
plt.ylabel("f'' at candidate")
plt.show()

▶ What you'll see: positive curvature gives a local minimum, negative gives a maximum, and zero is inconclusive.

### ✍️ Toy 4 · Hessian eigenvalues reveal saddle curvature

In many dimensions, different directions can curve differently. A Hessian with both positive and
negative eigenvalues exposes a saddle.

In [ ]:
import numpy as np                              # arrays and symmetric eigenvalues.

t4_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t4_H = np.array([[2.0, 0.0], [0.0, -4.0]])      # -> [[2.0, 0.0], [0.0, -4.0]]
print("Hessian:", t4_H.tolist())                # -> [[2.0, 0.0], [0.0, -4.0]]
t4_eigs = np.linalg.eigvalsh(t4_H)              # -> [-4.0, 2.0]
print("eigenvalues:", t4_eigs.tolist())         # -> [-4.0, 2.0]
t4_dir1 = np.array([1.0, 0.0])                  # -> [1.0, 0.0]
print("direction 1:", t4_dir1.tolist())         # -> [1.0, 0.0]
t4_dir2 = np.array([0.0, 1.0])                  # -> [0.0, 1.0]
print("direction 2:", t4_dir2.tolist())         # -> [0.0, 1.0]
t4_dir3 = np.array([1.0, 1.0]) / np.sqrt(2.0)   # -> [0.707, 0.707]
print("direction 3:", np.round(t4_dir3, 3).tolist())            # -> [0.707, 0.707]
t4_curv1 = float(t4_dir1 @ t4_H @ t4_dir1)      # -> 2.0
print("curvature direction 1:", t4_curv1)       # -> 2.0
t4_curv2 = float(t4_dir2 @ t4_H @ t4_dir2)      # -> -4.0
print("curvature direction 2:", t4_curv2)       # -> -4.0
t4_curv3 = float(t4_dir3 @ t4_H @ t4_dir3)      # -> -1.0
print("curvature direction 3:", round(t4_curv3, 3))             # -> -1.0
t4_curvatures = np.array([t4_curv1, t4_curv2, t4_curv3])         # -> [2.0, -4.0, -1.0]
print("sample curvatures:", np.round(t4_curvatures, 3).tolist()) # -> [2.0, -4.0, -1.0]
t4_mixed = bool(np.any(t4_eigs < 0.0) and np.any(t4_eigs > 0.0)) # -> True
print("mixed signs?", t4_mixed)                 # -> True
assert t4_mixed and t4_curv1 > 0.0 and t4_curv2 < 0.0

plt.figure(figsize=(4.4, 2.8))
plt.bar(["λ1", "λ2"], t4_eigs, color=["crimson", "seagreen"])
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 4 · mixed eigenvalue signs")
plt.ylabel("curvature")
plt.show()

▶ What you'll see: one eigenvalue bar is below zero and one is above zero, the saddle signature.

### ✍️ Toy 5 · Convex stationarity is globally optimal

For a convex differentiable function, a zero gradient makes the tangent lower bound flat at the
minimum value, so every checked point lies above it.

In [ ]:
import numpy as np                              # arrays and convex gaps.

t5_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t5_x_star = 2.0                                 # -> 2.0
print("stationary point:", t5_x_star)           # -> 2.0
t5_grad = 2.0 * (t5_x_star - 2.0)               # -> 0.0
print("gradient:", t5_grad)                     # -> 0.0
t5_f_star = (t5_x_star - 2.0) ** 2 + 1.0        # -> 1.0
print("f(x*):", t5_f_star)                      # -> 1.0
t5_grid = np.array([0.0, 1.0, 2.0, 3.0, 4.0])   # -> [0.0, 1.0, 2.0, 3.0, 4.0]
print("grid:", t5_grid.tolist())                # -> [0.0, 1.0, 2.0, 3.0, 4.0]
t5_values = (t5_grid - 2.0) ** 2 + 1.0          # -> [5.0, 2.0, 1.0, 2.0, 5.0]
print("f(grid):", t5_values.tolist())           # -> [5.0, 2.0, 1.0, 2.0, 5.0]
t5_gaps = t5_values - t5_f_star                 # -> [4.0, 1.0, 0.0, 1.0, 4.0]
print("global gaps:", t5_gaps.tolist())         # -> [4.0, 1.0, 0.0, 1.0, 4.0]
t5_all_above = bool(np.all(t5_gaps >= 0.0))     # -> True
print("all sampled points above x*?", t5_all_above) # -> True
assert t5_grad == 0.0 and t5_all_above

plt.figure(figsize=(4.4, 3.0))
plt.plot(t5_grid, t5_values, marker="o", color="seagreen", label="convex f")
plt.axhline(t5_f_star, color="black", linestyle="--", label="stationary value")
plt.scatter([t5_x_star], [t5_f_star], color="black", zorder=3)
plt.title("Toy 5 · zero gradient is global for convex f")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

▶ What you'll see: the whole sampled bowl sits at or above the stationary value `1.0`.

### ✍️ Toy 6 · Tiny gradients can hide distance

A flat scale can make the gradient very small even far from the minimizer. Objective gap and distance
still reveal that the point is not optimal.

In [ ]:
import numpy as np                              # arrays and scaled gradients.

t6_rng = np.random.default_rng(0)               # -> seed 0
print("rng seed:", 0)                           # -> 0
t6_scale = 0.0001                               # -> 0.0001
print("scale:", t6_scale)                       # -> 0.0001
t6_x = 8.0                                      # -> 8.0
print("point:", t6_x)                           # -> 8.0
t6_grad = 2.0 * t6_scale * t6_x                 # -> 0.0016
print("gradient:", t6_grad)                     # -> 0.0016
t6_value = t6_scale * t6_x ** 2                 # -> 0.0064
print("objective value:", t6_value)             # -> 0.0064
t6_opt_value = 0.0                              # -> 0.0
print("optimal value:", t6_opt_value)           # -> 0.0
t6_gap = t6_value - t6_opt_value                # -> 0.0064
print("objective gap:", t6_gap)                 # -> 0.0064
t6_grid = np.array([-8.0, -4.0, 0.0, 4.0, 8.0]) # -> [-8.0, -4.0, 0.0, 4.0, 8.0]
print("grid:", t6_grid.tolist())                # -> [-8.0, -4.0, 0.0, 4.0, 8.0]
t6_values = t6_scale * t6_grid ** 2             # -> [0.0064, 0.0016, 0.0, 0.0016, 0.0064]
print("f(grid):", t6_values.tolist())           # -> [0.0064, 0.0016, 0.0, 0.0016, 0.0064]
assert t6_grad < 0.002 and t6_gap == 0.0064

plt.figure(figsize=(4.4, 3.0))
plt.plot(t6_grid, t6_values, marker="o", color="purple")
plt.scatter([t6_x, 0.0], [t6_value, t6_opt_value], color=["crimson", "black"], zorder=3)
plt.title("Toy 6 · flat bowl, tiny slope")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

▶ What you'll see: the red point is far from zero even though its gradient is only `0.0016`.


## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, finite differences, gradients, Hessians, and eigenvalues.
import matplotlib.pyplot as plt  # load Matplotlib for curves, contours, and diagnostic plots.
np.random.seed(0)  # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Evaluate a simple objective

**Goal.** Compute values of $f(x)=(x-1)^2+2$, because optimality starts by comparing objective values near a candidate. We build it in 2 steps.

In [ ]:
x_b1 = np.array([-1.0, 0.0, 1.0, 2.0, 3.0])  # choose points around the expected minimizer x=1.
f_b1 = (x_b1 - 1.0) ** 2 + 2.0  # evaluate the quadratic bowl at each point.
print("x:", x_b1)
print("f(x):", f_b1)
assert f_b1[2] == 2.0

▶ What you'll see: the smallest listed value occurs at x=1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(x_b1, f_b1, marker="o", color="teal")
plt.title("Basic 1: quadratic values"); plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the sampled points form a U-shape.

👀 Takeaway: a minimum is a low objective value relative to nearby feasible points.

### Basic 2 — Compute a derivative value

**Goal.** Evaluate $f'(x)=2(x-1)$ for the same bowl, because the derivative reports local slope. We build it in 2 steps.

In [ ]:
x_b2 = np.array([0.0, 1.0, 2.0])  # inspect left, center, and right of the bowl.
grad_b2 = 2.0 * (x_b2 - 1.0)  # analytic derivative of (x-1)^2+2.
print("x:", x_b2)
print("f'(x):", grad_b2)
assert np.allclose(grad_b2, [-2.0, 0.0, 2.0])

▶ What you'll see: slope is negative left of the minimum, zero at it, and positive right of it.

In [ ]:
plt.figure(figsize=(4, 3))
plt.axhline(0, color="black", linewidth=0.8)
plt.plot(x_b2, grad_b2, marker="o", color="orange")
plt.title("Basic 2: derivative sign"); plt.xlabel("x"); plt.ylabel("f'(x)"); plt.show()

▶ What you'll see: the derivative crosses zero at the stationary candidate.

👀 Takeaway: zero derivative marks a candidate point, not automatically a final answer.

### Basic 3 — Take a tiny downhill step

**Goal.** Move opposite a nonzero derivative, because $-f'(x)$ is the one-dimensional descent direction. We build it in 2 steps.

In [ ]:
x_b3 = 3.0  # start to the right of the minimum.
grad_b3 = 2.0 * (x_b3 - 1.0)  # derivative at the start.
eta_b3 = 0.25  # choose a small step size.
x_new_b3 = x_b3 - eta_b3 * grad_b3  # gradient descent update.
print("old x:", x_b3, "gradient:", grad_b3, "new x:", x_new_b3)
assert x_new_b3 == 2.0

▶ What you'll see: the update moves left, toward the minimizer at x=1.

In [ ]:
old_val_b3 = (x_b3 - 1.0) ** 2 + 2.0
new_val_b3 = (x_new_b3 - 1.0) ** 2 + 2.0
print("f old:", old_val_b3, "f new:", new_val_b3)
assert new_val_b3 < old_val_b3

▶ What you'll see: the objective decreases after the downhill step.

In [ ]:
xs_b3 = np.linspace(0.5, 3.2, 200)
vals_curve_b3 = (xs_b3 - 1.0) ** 2 + 2.0
plt.figure(figsize=(4.5, 3))
plt.plot(xs_b3, vals_curve_b3, color="steelblue")
plt.scatter([x_b3, x_new_b3, 1.0], [old_val_b3, new_val_b3, 2.0], color=["crimson", "seagreen", "black"], zorder=3)
plt.annotate("", xy=(x_new_b3, new_val_b3), xytext=(x_b3, old_val_b3),
             arrowprops=dict(arrowstyle="->", color="seagreen", lw=2))
plt.title("Basic 3: downhill step on the bowl"); plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the update moves down the quadratic landscape toward the minimizer.

👀 Takeaway: a nonzero gradient gives a concrete local improvement direction.

### Basic 4 — Verify stationarity at a minimum

**Goal.** Check both derivative and value at $x=1$, because a true smooth interior minimum must be stationary. We build it in 2 steps.

In [ ]:
x_b4 = 1.0  # candidate for f(x)=(x-1)^2+2.
grad_b4 = 2.0 * (x_b4 - 1.0)  # derivative at the candidate.
f_b4 = (x_b4 - 1.0) ** 2 + 2.0  # objective value at the candidate.
print("gradient:", grad_b4, "value:", f_b4)
assert grad_b4 == 0.0 and f_b4 == 2.0

▶ What you'll see: the derivative vanishes at the bottom.

In [ ]:
neighbors_b4 = x_b4 + np.array([-0.2, 0.2])  # inspect nearby points on both sides.
neighbor_vals_b4 = (neighbors_b4 - 1.0) ** 2 + 2.0
print("neighbor values:", np.round(neighbor_vals_b4, 3))
assert np.all(neighbor_vals_b4 > f_b4)

▶ What you'll see: nearby points have larger values than the candidate.

In [ ]:
xs_b4 = np.linspace(0.5, 1.5, 200)
vals_b4_curve = (xs_b4 - 1.0) ** 2 + 2.0
plt.figure(figsize=(4.5, 3))
plt.plot(xs_b4, vals_b4_curve, color="teal")
plt.scatter(neighbors_b4, neighbor_vals_b4, color="orange", label="nearby points", zorder=3)
plt.scatter([x_b4], [f_b4], color="black", label="stationary minimum", zorder=4)
plt.title("Basic 4: stationary point with rising neighbors"); plt.xlabel("x"); plt.ylabel("f(x)")
plt.legend(); plt.show()

▶ What you'll see: the stationary candidate sits below both nearby points.

👀 Takeaway: stationarity plus rising neighbors is the local minimum pattern.

### Basic 5 — See a stationary maximum

**Goal.** Evaluate $f(x)=-x^2$ at zero, because a maximum also has zero derivative. We build it in 2 steps.

In [ ]:
x_b5 = 0.0  # stationary point of -x^2.
grad_b5 = -2.0 * x_b5  # derivative of -x^2.
curv_b5 = -2.0  # second derivative of -x^2.
print("gradient:", grad_b5, "second derivative:", curv_b5)
assert grad_b5 == 0.0 and curv_b5 < 0

▶ What you'll see: the first derivative is zero even though curvature is negative.

In [ ]:
near_b5 = np.array([-1.0, 0.0, 1.0])
vals_b5 = -near_b5 ** 2
print("values:", vals_b5)
plt.figure(figsize=(4, 3)); plt.plot(near_b5, vals_b5, marker="o", color="crimson")
plt.title("Basic 5: stationary maximum"); plt.show()

▶ What you'll see: x=0 is higher than its neighbors, so it is a maximum.

👀 Takeaway: zero derivative alone cannot distinguish minima from maxima.

### Basic 6 — Use the second derivative test

**Goal.** Classify three stationary candidates by curvature sign, because the second derivative is the 1-D local shape test. We build it in 2 steps.

In [ ]:
curv_b6 = np.array([2.0, -2.0, 0.0])  # bowl, hill, and cubic stationary curvatures.
classes_b6 = np.where(curv_b6 > 0, "min", np.where(curv_b6 < 0, "max", "inconclusive"))
print("curvatures:", curv_b6)
print("classes:", classes_b6)
assert classes_b6.tolist() == ["min", "max", "inconclusive"]

▶ What you'll see: positive curvature means min, negative means max, zero means no verdict.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["+", "-", "0"], curv_b6, color=["green", "red", "gray"])
plt.axhline(0, color="black", linewidth=0.8); plt.title("Basic 6: curvature signs"); plt.show()

▶ What you'll see: bars above, below, and on the axis match the three outcomes.

👀 Takeaway: the second derivative test only works when curvature is nonzero.

### Basic 7 — Build a 2-D gradient

**Goal.** Compute $\nabla f(x,y)$ for $f=(x-1)^2+2(y+2)^2$, because multivariable stationarity means every partial derivative is zero. We build it in 2 steps.

In [ ]:
point_b7 = np.array([2.0, -1.0])  # choose a 2-D point near the bowl minimum.
grad_b7 = np.array([2 * (point_b7[0] - 1.0), 4 * (point_b7[1] + 2.0)])  # partial derivatives.
print("point:", point_b7)
print("gradient:", grad_b7)
assert np.allclose(grad_b7, [2.0, 4.0])

▶ What you'll see: the gradient has one component per coordinate.

In [ ]:
plt.figure(figsize=(4, 3)); plt.quiver([point_b7[0]], [point_b7[1]], [grad_b7[0]], [grad_b7[1]], angles="xy", scale_units="xy", scale=1)
plt.xlim(0, 4); plt.ylim(-2, 3); plt.title("Basic 7: 2-D gradient vector"); plt.show()

▶ What you'll see: the vector points toward steepest local increase.

👀 Takeaway: multivariable descent moves opposite the whole gradient vector.

### Basic 8 — Build a Hessian matrix

**Goal.** Write the Hessian of a separable quadratic, because second-order optimality in many dimensions uses a matrix. We build it in 2 steps.

In [ ]:
H_b8 = np.array([[2.0, 0.0], [0.0, 4.0]])  # Hessian of (x-1)^2 + 2(y+2)^2.
print("Hessian:\n", H_b8)
print("shape:", H_b8.shape)
assert H_b8.shape == (2, 2)

▶ What you'll see: diagonal entries store coordinate-wise curvature.

In [ ]:
evals_b8 = np.linalg.eigvalsh(H_b8)  # eigenvalues of a symmetric Hessian.
print("eigenvalues:", evals_b8)
assert np.all(evals_b8 > 0)

▶ What you'll see: both eigenvalues are positive.

In [ ]:
theta_b8 = np.linspace(0, 2 * np.pi, 200)
circle_b8 = np.c_[np.cos(theta_b8), np.sin(theta_b8)]
quad_b8 = np.array([float(v @ H_b8 @ v) for v in circle_b8])
plt.figure(figsize=(4.5, 3))
plt.plot(theta_b8, quad_b8, color="seagreen")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 8: positive curvature in every direction"); plt.xlabel("direction angle"); plt.ylabel("vᵀHv"); plt.show()

▶ What you'll see: directional curvature stays above zero for the positive-definite Hessian.

👀 Takeaway: a positive-definite Hessian means the quadratic curves upward in every direction.

### Basic 9 — Detect a saddle by eigenvalues

**Goal.** Classify $f(x,y)=x^2-y^2$, because it has zero gradient but mixed curvature. We build it in 2 steps.

In [ ]:
point_b9 = np.array([0.0, 0.0])
grad_b9 = np.array([2 * point_b9[0], -2 * point_b9[1]])
H_b9 = np.array([[2.0, 0.0], [0.0, -2.0]])
evals_b9 = np.linalg.eigvalsh(H_b9)
print("gradient:", grad_b9, "eigenvalues:", evals_b9)
assert np.allclose(grad_b9, [0.0, 0.0]) and np.any(evals_b9 < 0) and np.any(evals_b9 > 0)

▶ What you'll see: stationarity plus mixed eigenvalues.

In [ ]:
dirs_b9 = np.eye(2)
curv_b9 = np.array([float(v @ H_b9 @ v) for v in dirs_b9])
print("x and y curvatures:", curv_b9)
assert np.allclose(curv_b9, [2.0, -2.0])

▶ What you'll see: one coordinate curves up and the other curves down.

In [ ]:
grid_b9 = np.linspace(-1.5, 1.5, 101)
X_b9, Y_b9 = np.meshgrid(grid_b9, grid_b9)
Z_b9 = X_b9 ** 2 - Y_b9 ** 2
plt.figure(figsize=(4.5, 3.5))
plt.contour(X_b9, Y_b9, Z_b9, levels=15, cmap="coolwarm")
plt.scatter([point_b9[0]], [point_b9[1]], color="black", zorder=3)
plt.quiver([0, 0], [0, 0], [1, 0], [0, 1], angles="xy", scale_units="xy", scale=1,
           color=["seagreen", "crimson"], width=0.008)
plt.axis("equal"); plt.title("Basic 9: saddle eigen-directions"); plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: contours rise along one eigen-direction and fall along the other.

👀 Takeaway: a Hessian with both signs is a saddle certificate.

### Basic 10 — Check convex global optimality

**Goal.** Confirm that a convex stationary point is globally optimal, because convexity upgrades local stationarity. We build it in 2 steps.

In [ ]:
xs_b10 = np.linspace(-2, 4, 301)
vals_b10 = (xs_b10 - 1.0) ** 2 + 2.0
xstar_b10 = 1.0
fstar_b10 = 2.0
grad_b10 = 2.0 * (xstar_b10 - 1.0)
print("gradient at x*:", grad_b10, "min sampled value:", round(float(np.min(vals_b10)), 3))
assert grad_b10 == 0.0 and abs(np.min(vals_b10) - fstar_b10) < 1e-12

▶ What you'll see: the sampled minimum agrees with the stationary point.

In [ ]:
gaps_b10 = vals_b10 - fstar_b10
print("smallest gap:", round(float(np.min(gaps_b10)), 6))
plt.figure(figsize=(4, 3)); plt.plot(xs_b10, gaps_b10, color="seagreen")
plt.title("Basic 10: global nonnegative gaps"); plt.xlabel("x"); plt.ylabel("f(x)-f(x*)"); plt.show()

▶ What you'll see: every plotted gap is nonnegative.

👀 Takeaway: for convex differentiable functions, $\nabla f(x^\star)=0$ is enough for global optimality.

## 🟡 Easy

### Easy 1 — Finite-difference a gradient

**Goal.** Approximate a derivative numerically, because optimizers sometimes verify analytic gradients by finite differences. We build it in 3 steps.

In [ ]:
x_e1 = 1.5  # point where we check the derivative of (x-1)^2+2.
h_e1 = 1e-4  # small symmetric finite-difference step.
f_plus_e1 = (x_e1 + h_e1 - 1.0) ** 2 + 2.0
f_minus_e1 = (x_e1 - h_e1 - 1.0) ** 2 + 2.0
print("f(x+h):", round(f_plus_e1, 6), "f(x-h):", round(f_minus_e1, 6))

▶ What you'll see: two nearby function evaluations around the target point.

In [ ]:
fd_grad_e1 = (f_plus_e1 - f_minus_e1) / (2 * h_e1)
true_grad_e1 = 2.0 * (x_e1 - 1.0)
print("finite-diff grad:", round(fd_grad_e1, 6), "true grad:", true_grad_e1)
assert abs(fd_grad_e1 - true_grad_e1) < 1e-8

▶ What you'll see: the finite-difference estimate matches the analytic derivative.

In [ ]:
plt.figure(figsize=(4, 3)); plt.scatter([x_e1 - h_e1, x_e1, x_e1 + h_e1], [f_minus_e1, (x_e1 - 1) ** 2 + 2, f_plus_e1])
plt.title("Easy 1: local samples for slope"); plt.show()

▶ What you'll see: nearby samples define the local slope estimate.

👀 Takeaway: finite differences approximate gradients by measuring tiny symmetric changes in the objective.

### Easy 2 — Classify stationary points from formulas

**Goal.** Find and classify candidates for three 1-D functions, because stationarity must be followed by curvature. We build it in 3 steps.

In [ ]:
names_e2 = np.array(["(x-1)^2+2", "-x^2", "x^4"])
candidates_e2 = np.array([1.0, 0.0, 0.0])
curv_e2 = np.array([2.0, -2.0, 0.0])
print("candidates:", list(zip(names_e2, candidates_e2)))

▶ What you'll see: each function has a stationary candidate.

In [ ]:
classes_e2 = np.where(curv_e2 > 0, "strict min", np.where(curv_e2 < 0, "strict max", "needs higher-order check"))
for name_e2, curv_i_e2, class_i_e2 in zip(names_e2, curv_e2, classes_e2):
    print(name_e2, "curvature", curv_i_e2, "->", class_i_e2)
assert classes_e2[2] == "needs higher-order check"

▶ What you'll see: $x^4$ has zero second derivative, so the usual second-derivative test is inconclusive.

In [ ]:
xs_e2 = np.linspace(-1.5, 1.5, 200)
plt.figure(figsize=(5, 3)); plt.plot(xs_e2, xs_e2 ** 4, label="x^4")
plt.plot(xs_e2, -xs_e2 ** 2, label="-x^2"); plt.legend(); plt.title("Easy 2: zero curvature can still be a min"); plt.show()

▶ What you'll see: $x^4$ is still a minimum even though its second derivative at zero is 0.

👀 Takeaway: positive curvature is sufficient for a strict local minimum, but zero curvature is not automatically failure.

### Easy 3 — Check Hessian positive definiteness

**Goal.** Test a symmetric Hessian by eigenvalues, because positive definiteness is the multivariable curvature condition. We build it in 3 steps.

In [ ]:
H_e3 = np.array([[4.0, 1.0], [1.0, 3.0]])  # a symmetric Hessian candidate.
evals_e3 = np.linalg.eigvalsh(H_e3)
print("H:\n", H_e3)
print("eigenvalues:", np.round(evals_e3, 3))
assert np.all(evals_e3 > 0)

▶ What you'll see: both eigenvalues are positive.

In [ ]:
theta_e3 = np.linspace(0, 2 * np.pi, 12, endpoint=False)
dirs_e3 = np.c_[np.cos(theta_e3), np.sin(theta_e3)]
quad_e3 = np.array([float(v @ H_e3 @ v) for v in dirs_e3])
print("min sampled directional curvature:", round(float(np.min(quad_e3)), 3))
assert np.min(quad_e3) > 0

▶ What you'll see: sampled directions all have positive quadratic curvature.

In [ ]:
plt.figure(figsize=(4, 3)); plt.plot(theta_e3, quad_e3, marker="o", color="teal")
plt.title("Easy 3: curvature by direction"); plt.xlabel("angle"); plt.ylabel("vᵀHv"); plt.show()

▶ What you'll see: the directional curvature curve stays above zero.

👀 Takeaway: positive eigenvalues certify upward curvature in every direction, not just coordinate axes.

### Easy 4 — Visualize a saddle contour

**Goal.** Plot $x^2-y^2$, because saddles are easiest to understand as opposite curvature directions. We build it in 3 steps.

In [ ]:
grid_e4 = np.linspace(-2, 2, 101)
X_e4, Y_e4 = np.meshgrid(grid_e4, grid_e4)
Z_e4 = X_e4 ** 2 - Y_e4 ** 2
print("grid shape:", Z_e4.shape, "center value:", Z_e4[50, 50])
assert Z_e4[50, 50] == 0.0

▶ What you'll see: a square grid of objective values centered at the stationary point.

In [ ]:
H_e4 = np.array([[2.0, 0.0], [0.0, -2.0]])
evals_e4 = np.linalg.eigvalsh(H_e4)
print("saddle eigenvalues:", evals_e4)
assert np.any(evals_e4 < 0) and np.any(evals_e4 > 0)

▶ What you'll see: the Hessian is indefinite.

In [ ]:
plt.figure(figsize=(4.5, 3.5)); plt.contour(X_e4, Y_e4, Z_e4, levels=15, cmap="coolwarm")
plt.scatter([0], [0], color="black"); plt.axis("equal"); plt.title("Easy 4: saddle contour"); plt.show()

▶ What you'll see: contour lines pinch through the origin instead of surrounding a bowl bottom.

👀 Takeaway: mixed Hessian signs mean nearby downhill directions exist despite zero gradient.

### Easy 5 — Compare local and global minima in a nonconvex curve

**Goal.** Inspect a nonconvex quartic, because positive curvature at one point only gives a local statement without convexity. We build it in 3 steps.

In [ ]:
xs_e5 = np.linspace(-2.5, 2.5, 501)
vals_e5 = xs_e5 ** 4 - 3 * xs_e5 ** 2 + 0.5 * xs_e5  # a nonconvex quartic.
grad_e5 = 4 * xs_e5 ** 3 - 6 * xs_e5 + 0.5
print("value range:", round(float(np.min(vals_e5)), 3), "to", round(float(np.max(vals_e5)), 3))

▶ What you'll see: a curve with multiple valleys and a hill.

In [ ]:
sign_changes_e5 = np.where(np.diff(np.sign(grad_e5)) != 0)[0]
crit_x_e5 = xs_e5[sign_changes_e5]
crit_vals_e5 = crit_x_e5 ** 4 - 3 * crit_x_e5 ** 2 + 0.5 * crit_x_e5
print("approx critical x:", np.round(crit_x_e5, 3))
print("critical values:", np.round(crit_vals_e5, 3))
assert len(crit_x_e5) >= 3

▶ What you'll see: several stationary candidates appear.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(xs_e5, vals_e5, color="purple")
plt.scatter(crit_x_e5, crit_vals_e5, color="black"); plt.title("Easy 5: nonconvex local candidates"); plt.show()

▶ What you'll see: at least one stationary valley is only local, not necessarily global.

👀 Takeaway: without convexity, second-order conditions describe local shape rather than global optimality.

## 🔴 Advanced

### Advanced 1 — Newton step from gradient and Hessian

**Goal.** Compute a Newton update on a quadratic, because Newton's method solves the local second-order model. We build it in 4 steps.

In [ ]:
A_a1 = np.array([[4.0, 1.0], [1.0, 3.0]])  # positive-definite quadratic Hessian.
b_a1 = np.array([-1.0, 2.0])  # linear term in f(x)=0.5 xᵀAx + bᵀx.
x_a1 = np.array([2.0, -2.0])  # starting point.
grad_a1 = A_a1 @ x_a1 + b_a1  # gradient of the quadratic.
print("gradient:", grad_a1)
assert np.allclose(grad_a1, [5.0, -2.0])

▶ What you'll see: the current point is not stationary.

In [ ]:
step_a1 = -np.linalg.solve(A_a1, grad_a1)  # Newton step solves H step = -grad.
x_new_a1 = x_a1 + step_a1
print("Newton step:", np.round(step_a1, 3), "new x:", np.round(x_new_a1, 3))

▶ What you'll see: the step jumps directly to the quadratic minimizer.

In [ ]:
grad_new_a1 = A_a1 @ x_new_a1 + b_a1
print("new gradient:", np.round(grad_new_a1, 10))
assert np.allclose(grad_new_a1, [0.0, 0.0])

▶ What you'll see: the new point is stationary up to numerical precision.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["||grad before||", "||grad after||"], [np.linalg.norm(grad_a1), np.linalg.norm(grad_new_a1)], color=["orange", "green"])
plt.title("Advanced 1: Newton kills quadratic gradient"); plt.show()

▶ What you'll see: the gradient norm collapses to zero after one Newton step.

👀 Takeaway: for a positive-definite quadratic, Newton's stationarity equation reaches the minimizer in one exact solve.

### Advanced 2 — Indefinite Hessian exposes a negative-curvature direction

**Goal.** Find a direction that lowers the objective from a stationary saddle, because indefinite curvature invalidates minimum claims. We build it in 4 steps.

In [ ]:
H_a2 = np.array([[3.0, 0.0], [0.0, -1.0]])  # Hessian with mixed signs.
evals_a2, evecs_a2 = np.linalg.eigh(H_a2)
neg_idx_a2 = int(np.argmin(evals_a2))
neg_dir_a2 = evecs_a2[:, neg_idx_a2]
print("eigenvalues:", evals_a2, "negative direction:", neg_dir_a2)
assert evals_a2[neg_idx_a2] < 0

▶ What you'll see: the eigenvector for the negative eigenvalue is a downhill curvature direction.

In [ ]:
t_a2 = 0.2
change_a2 = 0.5 * t_a2 ** 2 * float(neg_dir_a2 @ H_a2 @ neg_dir_a2)
print("quadratic change along negative direction:", round(change_a2, 4))
assert change_a2 < 0

▶ What you'll see: moving a little along that eigenvector decreases the quadratic model.

In [ ]:
dir_pos_a2 = evecs_a2[:, int(np.argmax(evals_a2))]
change_pos_a2 = 0.5 * t_a2 ** 2 * float(dir_pos_a2 @ H_a2 @ dir_pos_a2)
print("change along positive direction:", round(change_pos_a2, 4))
assert change_pos_a2 > 0

▶ What you'll see: another direction increases the model, confirming saddle behavior.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["negative eigendir", "positive eigendir"], [change_a2, change_pos_a2], color=["crimson", "seagreen"])
plt.axhline(0, color="black", linewidth=0.8); plt.title("Advanced 2: curvature directions"); plt.xticks(rotation=12); plt.show()

▶ What you'll see: one small move has negative predicted change and the other positive.

👀 Takeaway: a single negative Hessian eigenvalue proves a stationary point is not a local minimum.

### Advanced 3 — Use the tangent inequality for convexity

**Goal.** Numerically check the convex tangent lower-bound property, because it explains why stationarity is globally sufficient for convex objectives. We build it in 4 steps.

In [ ]:
x0_a3 = 0.0  # tangent point for f(x)=exp(x).
y_a3 = np.linspace(-2, 2, 101)  # points to compare against the tangent.
f_x0_a3 = np.exp(x0_a3)
grad_x0_a3 = np.exp(x0_a3)
tangent_a3 = f_x0_a3 + grad_x0_a3 * (y_a3 - x0_a3)
print("tangent at x0 has intercept:", f_x0_a3, "slope:", grad_x0_a3)
assert f_x0_a3 == 1.0 and grad_x0_a3 == 1.0

▶ What you'll see: the tangent line to exp(x) at zero is $1+x$.

In [ ]:
f_y_a3 = np.exp(y_a3)
gaps_a3 = f_y_a3 - tangent_a3
print("minimum tangent gap:", round(float(np.min(gaps_a3)), 6))
assert np.min(gaps_a3) >= -1e-12

▶ What you'll see: every sampled point lies on or above the tangent line.

In [ ]:
stationary_grad_a3 = 2 * (1.0 - 1.0)  # gradient of convex (x-1)^2+2 at x=1.
print("stationary convex gradient:", stationary_grad_a3)
assert stationary_grad_a3 == 0.0

▶ What you'll see: at a convex stationary point, the tangent lower bound becomes a horizontal global bound.

In [ ]:
plt.figure(figsize=(4.5, 3)); plt.plot(y_a3, f_y_a3, label="exp(x)"); plt.plot(y_a3, tangent_a3, label="tangent", linestyle="--")
plt.legend(); plt.title("Advanced 3: convex tangent lower bound"); plt.show()

▶ What you'll see: the tangent stays below the convex curve.

👀 Takeaway: convexity makes first-order information global through the tangent lower-bound inequality.

### Advanced 4 — Track gradient norm and objective decrease together

**Goal.** Run gradient descent on a flat quadratic and monitor two diagnostics, because small gradients alone can hide slow progress. We build it in 4 steps.

In [ ]:
x_a4 = 10.0
eta_a4 = 1000.0  # safe for f(x)=1e-4 x^2 because curvature is only 2e-4.
xs_a4 = [x_a4]
vals_a4 = [1e-4 * x_a4 ** 2]
grads_a4 = [2e-4 * x_a4]
print("start value:", vals_a4[0], "start gradient:", grads_a4[0])
assert round(vals_a4[0], 3) == 0.01 and round(grads_a4[0], 3) == 0.002

▶ What you'll see: the initial gradient is tiny even at x=10.

In [ ]:
for _a4 in range(8):
    grad_now_a4 = 2e-4 * xs_a4[-1]
    x_next_a4 = xs_a4[-1] - eta_a4 * grad_now_a4
    xs_a4.append(x_next_a4)
    vals_a4.append(1e-4 * x_next_a4 ** 2)
    grads_a4.append(2e-4 * x_next_a4)
print("final x:", round(xs_a4[-1], 3), "final value:", round(vals_a4[-1], 6))
assert vals_a4[-1] < vals_a4[0]

▶ What you'll see: objective progress occurs even though gradients are numerically small throughout.

In [ ]:
print("gradient norms:", np.round(np.abs(grads_a4), 6))
print("values:", np.round(vals_a4, 6))

▶ What you'll see: both diagnostics shrink, but on very different numerical scales.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(vals_a4, marker="o", label="objective"); plt.plot(np.abs(grads_a4), marker="s", label="|gradient|")
plt.yscale("log"); plt.legend(); plt.title("Advanced 4: monitor more than one scale"); plt.show()

▶ What you'll see: log scale reveals simultaneous decay of objective value and gradient norm.

👀 Takeaway: approximate optimality checks should consider scale, curvature, and objective progress, not just raw gradient size.

### Advanced 5 — Compare local sufficient and global convex conditions

**Goal.** Contrast a nonconvex local minimum with a convex global minimum, because positive-definite curvature is local unless convexity holds. We build it in 4 steps.

In [ ]:
xs_a5 = np.linspace(-3, 3, 1201)
nonconvex_a5 = (xs_a5 ** 2 - 1) ** 2 + 0.2 * xs_a5  # tilted double-well objective.
convex_a5 = (xs_a5 - 0.5) ** 2  # simple convex objective.
print("nonconvex min approx:", round(float(xs_a5[np.argmin(nonconvex_a5)]), 3))
print("convex min approx:", round(float(xs_a5[np.argmin(convex_a5)]), 3))

▶ What you'll see: the nonconvex curve has a best well, while the convex curve has one global bowl bottom.

In [ ]:
grad_non_a5 = 4 * xs_a5 * (xs_a5 ** 2 - 1) + 0.2
crit_idx_a5 = np.where(np.diff(np.sign(grad_non_a5)) != 0)[0]
crit_x_a5 = xs_a5[crit_idx_a5]
curv_crit_a5 = 12 * crit_x_a5 ** 2 - 4
print("nonconvex critical x:", np.round(crit_x_a5, 3))
print("curvatures there:", np.round(curv_crit_a5, 3))
assert np.any(curv_crit_a5 > 0)

▶ What you'll see: positive curvature identifies local minima among nonconvex stationary points.

In [ ]:
local_min_x_a5 = crit_x_a5[curv_crit_a5 > 0]
local_vals_a5 = (local_min_x_a5 ** 2 - 1) ** 2 + 0.2 * local_min_x_a5
print("local min values:", np.round(local_vals_a5, 3))
assert len(local_min_x_a5) == 2

▶ What you'll see: both wells are local minima, but their objective values differ.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(xs_a5, nonconvex_a5, label="nonconvex"); plt.plot(xs_a5, convex_a5, label="convex")
plt.scatter(local_min_x_a5, local_vals_a5, color="black", zorder=3); plt.legend(); plt.title("Advanced 5: local vs global certificates"); plt.show()

▶ What you'll see: positive curvature marks both nonconvex wells, while convexity would rule out the higher local trap.

👀 Takeaway: Hessian positive definiteness is a local sufficient condition; convexity is what turns stationarity into a global certificate.